# 02. Procesamiento en la nube y muestreo estratificado
Objetivo de este notebook:
En lugar de extraer datos aleatorios, programaremos la lógica del filtro temporal de MapBiomas directamente en los servidores de Google. Evaluaremos una ventana móvil (2019-2020-2021) para detectar píxeles con comportamiento inconsistente (Bosque -> Suelo Desnudo -> Bosque).


Finalmente, aplicaremos un muestreo estratificado para extraer exactamente 1.000 muestras normales y 1.000 muestras espurias, resolviendo el desbalanceo de clases antes de entrenar nuestro modelo de Machine Learning.

In [6]:
import ee
import pandas as pd
import os

ee.Initialize()

# 1. Definir nuestra área de estudio en el Chaco
aoi = ee.Geometry.Rectangle([-63.0, -27.5, -61.0, -26.0])

# 2. Cargar los catálogos
mapbiomas = ee.Image('projects/mapbiomas-argentina/assets/LAND-COVER/COLLECTION-2/GENERAL/CLASSIFICATION/FINAL_CLASSIFICATION/CHACO/CHACO-FINAL-v1')
clima_era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

# 3. Seleccionar la "Ventana Móvil" de 3 años
mapa_2019 = mapbiomas.select('classification_2019')
mapa_2020 = mapbiomas.select('classification_2020')
mapa_2021 = mapbiomas.select('classification_2021')

# 4. Clima de 2020
clima_2020 = clima_era5.filterDate('2020-01-01', '2020-12-31').mean().select(['temperature_2m', 'total_precipitation_sum'])

Calculamos la variable objetivo (es_espurio) directamente en los servidores de Google usando álgebra de mapas.

In [7]:
# LÓGICA ESPURIA EN LA NUBE: 
# Es 1 (Verdadero) SI el año 2019 es igual a 2021, Y el 2019 es distinto al 2020.
es_espurio = mapa_2019.eq(mapa_2021).And(mapa_2019.neq(mapa_2020)).rename('es_espurio')

# Unimos todas las bandas, incluyendo nuestra nueva banda matemática
imagen_multitemporal = mapa_2019.addBands(mapa_2020).addBands(mapa_2021).addBands(clima_2020).addBands(es_espurio)

# MUESTREO ESTRATIFICADO: Pedimos exactamente 1000 puntos de clase 0 y 1000 puntos de clase 1
print("Buscando y extrayendo muestras perfectamente balanceadas en la nube...")
extraccion_estratificada = imagen_multitemporal.stratifiedSample(
    numPoints=1000, 
    classBand='es_espurio', 
    region=aoi, 
    scale=30,
    geometries=False
)

Buscando y extrayendo muestras perfectamente balanceadas en la nube...


Como ahora solo traemos 2.000 puntos exactos, podemos usar getInfo() sin saturar la memoria y guardar nuestro CSV listo para el modelo.

In [ ]:
# Crear carpeta si no existe
os.makedirs('data', exist_ok=True)
ruta_limpia = 'data/dataset_etiquetado.csv'

# Descargar la información 
info = extraccion_estratificada.getInfo()
datos = [p.get('properties', {}) for p in info.get('features', [])]

# Convertir a Pandas y guardar
df_temporal = pd.DataFrame(datos)
df_temporal.to_csv(ruta_limpia, index=False)

print("¡Extracción exitosa!")
print("Distribución de la muestra (Perfectamente balanceada):")
print(df_temporal['es_espurio'].value_counts())
display(df_temporal.tail())

¡Extracción exitosa!
Distribución de la muestra (Perfectamente balanceada):
es_espurio
0    1000
1    1000
Name: count, dtype: int64


,classification_2019,classification_2020,classification_2021,es_espurio,temperature_2m,total_precipitation_sum
1995,15,19,15,1,296.410291,0.059977
1996,15,19,15,1,295.717747,0.061225
1997,15,19,15,1,295.820594,0.069567
1998,4,19,4,1,296.157602,0.065041
1999,4,19,4,1,296.305425,0.058188
